## Création d'un DataLoader pour un dataset d'images

### Rappel :

Un `Dataset` (structure utilisée par exemple par PyTorch) est une __classe__ possédant :

- une méthode `__init__(self, ...)` permettant de spécifier où se trouvent les données que vous souhaitez manipuler, 
- une méthode `__len__(self)`  permettant de connaitre le nombre d'objets de votre jeu de données,
- une méthode `__getitem__(self, idx)` permettant d'obtenir l'élément numéro `idx` (i.e., le idx-ème élément) du jeu de données.

1) Commencez par télécharger le jeu de données `tf_flowers` de TensorFlow

In [22]:
import os
from pathlib import Path
import requests
import tarfile

url = 'http://download.tensorflow.org/example_images/flower_photos.tgz'
output_dir = Path('./image_dataset/')
tgz_file = output_dir / 'flower_photos.tgz'

os.makedirs(output_dir, exist_ok=True)

with open(tgz_file, 'wb') as f:
    f.write(requests.get(url).content)

2) Décompressez l'archive téléchargée dans le même dossier et supprimez l'archive :

In [23]:
with tarfile.open(tgz_file, 'r:gz') as tar:
    tar.extractall(output_dir, filter='data')

os.remove(tgz_file)

Vous devriez obtenir un dossier `flower_photos/` contenant 5 sous-dossiers :
- un dossier `daisy/` contenant des images de marguerites,
- un dossier `dandelion/` contenant des images de pissenlits,
- un dossier `roses/` contenant des images de roses,
- un dosser `sunflowers/` contenant des images de tournesols,
- un dossier `tulips/` contenant des images de tulipes.


3. Dans un fichier Python, créez une classe `CustomImageDataset` qui définit un Dataset.

Cette classe doit prendre en paramèters `image_paths`, les chemins vers les images du jeu de données. On peut également lui fournir `transform` (la valeur par défaut est `None`), une instance de `albumentations.Compose`. La classe `CustomImageDataset` doit appliquer la transformation indiquée par `transform` le cas échéant.

In [24]:
import numpy as np
import cv2
from pathlib import Path

IMG_SIZE = (128, 128)
DATA_DIR = Path("./image_dataset/flower_photos")
CLASSES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

class CustomImageDataset:
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(str(self.image_paths[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, IMG_SIZE)
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        return image


4. Testez votre implémentation dans un nouveau fichier `main.py` en utilisant le jeu de données `tf_flowers`.

In [25]:
image_paths = []
for cls in CLASSES:
    image_paths.extend((DATA_DIR / cls).glob("*.jpg"))

dataset = CustomImageDataset(image_paths)
print(f"Nombre d'images : {len(dataset)}")
img = dataset[0]
print(f"Shape de la première image : {img.shape}")


Nombre d'images : 3670
Shape de la première image : (128, 128, 3)


5. Créez maintenant dans un troisième fichier une nouvelle classe `CustomLabelledImageLoader` reprenant les fonctionnalités de la classe `CustomImage`. Cette nouvelle classe doit également permettre de retourner l'étiquette (_label_) associé à une image.

**Aide :**  à la place de `image_paths`, on peut lui donner `label_mapping`, un dictionnaire dont les clés sont les chemins vers les images du jeu de données et les valeurs sont leur label.

In [26]:
class CustomLabelledImageDataset(CustomImageDataset):
    def __init__(self, label_mapping, transform=None, target_transform=None):
        super().__init__(list(label_mapping.keys()), transform)
        self.label_mapping = label_mapping
        self.target_transform = target_transform

    def __getitem__(self, idx):
        image = super().__getitem__(idx)
        label = self.label_mapping[self.image_paths[idx]]
        if self.target_transform is not None:
            label = self.target_transform.transform([label])[0]
        return image, label


6. Testez votre implémentation de cette classe dans votre fichier `main.py`.

In [27]:
label_mapping = {}
for cls in CLASSES:
    for path in (DATA_DIR / cls).glob("*.jpg"):
        label_mapping[path] = cls

labelled_dataset = CustomLabelledImageDataset(label_mapping)
img, label = labelled_dataset[0]
print(f"Shape image : {img.shape}, label : {label}")


Shape image : (128, 128, 3), label : daisy


7. Ajoutez à votre classe `CustomLabelledImage` la possibilité d'encoder les étiquettes via la librairie `scikit-learn`.

**Aide :** on peut fournir un nouveau paramètre `target_transform` de type `Callable` à la classe. Celui-ci peut être un `LabelBinarizer` ou un `LabelEncoder` (valeur par défaut `None`) et permet d'encoder le `label` le cas échéant.

8. Testez cette nouvelle fonctionnalité dans votre fichier `main.py`.

In [28]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(CLASSES)

encoded_dataset = CustomLabelledImageDataset(label_mapping, target_transform=le)
img, label = encoded_dataset[0]
print(f"Shape image : {img.shape}, label encodé : {label}")


Shape image : (128, 128, 3), label encodé : 0


9. Créez maintenant un générateur permettant d'itérer sur votre `CustomDataset` en générant des _batches_ de taille `batch_size` et testez la fonctionnalité dans votre fichier `main.py`.

Le prototype de la fonction doit être :
```python
def my_custom_generator(
    dataset: CustomImageDataset | CustomLabelledImageDataset,
    batch_size: int,
    shuffle: bool = False
) -> Iterator[tuple[np.ndarray, np.ndarray] | np.ndarray]:
    #TODO
```

In [29]:
import random

def my_custom_generator(dataset, batch_size, shuffle=False):
    indices = list(range(len(dataset)))
    if shuffle:
        random.shuffle(indices)
    for i in range(0, len(indices), batch_size):
        batch = [dataset[idx] for idx in indices[i:i + batch_size]]
        if isinstance(batch[0], tuple):
            yield np.array([b[0] for b in batch]), np.array([b[1] for b in batch])
        else:
            yield np.array(batch)

# Test
for i, batch in enumerate(my_custom_generator(dataset, batch_size=32)):
    print(f"Batch {i} : shape={batch.shape}")
    if i == 2:
        break


Batch 0 : shape=(32, 128, 128, 3)
Batch 1 : shape=(32, 128, 128, 3)
Batch 2 : shape=(32, 128, 128, 3)


Un Data Loader est une classe agissant comme un itérateur pour charger les données par paquets (batches). Elle possède :

- une méthode `__init__(self, dataset, batch_size, ...)` pour lier le Dataset et définir la taille des lots,
- une méthode `__iter__(self)` permettant d'initialiser (ou réinitialiser) le processus d'itération au début d'une boucle,
- une méthode `__next__(self)` permettant de retourner le prochain batch de données ou de signaler la fin du parcours,
- une méthode `__len__(self)` permettant de connaître le nombre total de batches à parcourir pour une époque complète.

10. Dans un fichier Python, créez une classe `CustomDataLoader` qui définit un Data Loader.

In [30]:
class CustomDataLoader:
    def __init__(self, dataset, batch_size, shuffle=False):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self._indices = []
        self._current = 0

    def __len__(self):
        return (len(self.dataset) + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        self._indices = list(range(len(self.dataset)))
        if self.shuffle:
            random.shuffle(self._indices)
        self._current = 0
        return self

    def __next__(self):
        if self._current >= len(self.dataset):
            raise StopIteration
        batch = [self.dataset[idx] for idx in self._indices[self._current:self._current + self.batch_size]]
        self._current += self.batch_size
        if isinstance(batch[0], tuple):
            return np.array([b[0] for b in batch]), np.array([b[1] for b in batch])
        return np.array(batch)


11. Testez votre implémentation de cette classe dans votre fichier `main.py`.

In [ ]:
loader = CustomDataLoader(labelled_dataset, batch_size=32, shuffle=True)
print(f"Nombre de batches : {len(loader)}")

for i, (images, labels) in enumerate(loader):
    print(f"Batch {i} : images={images.shape}, labels={labels.shape}")
    if i == 2:
        break


Nombre de batches : 115
Batch 0 : images=(32, 128, 128, 3), labels=(32,)
Batch 1 : images=(32, 128, 128, 3), labels=(32,)
Batch 2 : images=(32, 128, 128, 3), labels=(32,)


: 